# 🎯 Masterclass 02: Supervised Classification with Regularized Log-Loss
This notebook details probability classification pipelines:

1. **Project 1 (Theory Scratch)**: A custom binary Logistic Regression classifier with log-loss gradient descent.
2. **Project 2 (Applied Industry)**: A credit default pipeline addressing class imbalance with SMOTE and threshold tuning.


## 🧠 Project 1: From-Scratch Vectorized Classifier


In [ ]:
import numpy as np
import pandas as pd

class LogisticRegressionScratch:
    def __init__(self, lr=0.05, epochs=1000, reg_strength=0.1):
        self.lr = lr
        self.epochs = epochs
        self.reg = reg_strength
        self.w = None
        self.b = None

    def _sigmoid(self, z):
        z_clipped = np.clip(z, -25.0, 25.0)
        return 1.0 / (1.0 + np.exp(-z_clipped))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0

        for epoch in range(self.epochs):
            z = np.dot(X, self.w) + self.b
            p = self._sigmoid(z)
            dw = (1 / n_samples) * np.dot(X.T, (p - y)) + (self.reg / n_samples) * self.w
            db = (1 / n_samples) * np.sum(p - y)
            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict_proba(self, X):
        return self._sigmoid(np.dot(X, self.w) + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


## 🧪 Project 2: Production Credit Risk Pipeline


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE

# Generate imbalanced dataset
np.random.seed(42)
X_sim = np.random.randn(1000, 5)
y_sim = np.random.choice([0, 1], size=1000, p=[0.95, 0.05])

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_sim, y_sim)

X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)
probs = model.predict_proba(X_test)[:, 1]
print('ROC AUC Score:', roc_auc_score(y_test, probs))
